# Week 6 — Automated Data Collection

**Advanced Data Analysis · M. Fuat Kına**
`fuatkina@gmail.com` · `mkina@ku.edu.tr`

Two ways of building a dataset that does not exist yet.

| Part | Topic | Case |
|---|---|---|
| I | Web scraping with Scrapy selectors | Real estate listings on emlakjet |
| II | Reading an API | Anatomy of a request, JSON, pagination |
| III | Use case + exercise | YouTube Data API v3 |

Parts of Part I follow: https://app.datacamp.com/learn/courses/web-scraping-with-python

---
### Before you start

```
pip install scrapy requests pandas
```

Two habits worth forming today:

1. **Save the raw response** before you parse it. A parse can be rewritten; a page you no longer have cannot be re-downloaded as it was.
2. **Never hard-code a secret.** API keys live in an environment variable or an untracked file — not in a notebook you push to GitHub.

---
# Part I — Web scraping

## 1. One page, one Selector

In [1]:
from scrapy import Selector
import requests

# A real User-Agent is basic courtesy: it tells the server who is asking.
HEADERS = {"User-Agent": "ADA-course-notebook/1.0 (academic use; mkina@ku.edu.tr)"}

url = 'https://www.emlakjet.com/'

# Download the raw HTML the server sends back
html = requests.get(url, headers=HEADERS, timeout=20).content

# Wrap it in a Selector: a string becomes a navigable tree
sel = Selector(text=html)

print("You have found:", len(sel.xpath('//*')), "elements in your html!")

You have found: 3557 elements in your html!


In [2]:
# What are we actually holding?
print(type(html))   # bytes  — the raw document
print(type(sel))    # Selector — the parsed tree
html[:300]

<class 'bytes'>
<class 'scrapy.selector.unified.Selector'>


b'<!DOCTYPE html><html lang="tr" data-theme="emlakjet" class="lexend_94a73695-module__JaR9tq__variable lexendlatinext_372517df-module__TbR5ba__variable lexendvietnamese_4c56b2e2-module__q1Hgcq__variable" style="--emlakjet-main-header-offset:4rem"><head><meta charSet="utf-8"/><meta name="viewport" cont'

## 2. From an XPath to a list of strings

`.extract()` returns a list of matches — always a list, even when there is one match,
and an empty list when the path matched nothing. An empty list is the single most
common symptom of a broken scraper: the request succeeded, the path did not.

In [3]:
xpath_for_titles = '//h1'
sel.xpath(xpath_for_titles).extract()

['<h1 class="mb-6 text-center text-2xl leading-tight font-semibold text-(--color-hero-text) md:text-[32px]">Akıllı İlanlar, Doğru Kararlar</h1>']

In [4]:
len(sel.xpath(xpath_for_titles).extract())

1

In [5]:
# Adding text() asks for the text carried by the element, not the element itself
xpath_for_titles = '//h1/text()'
titles = sel.xpath(xpath_for_titles).extract()
titles

['Akıllı İlanlar, Doğru Kararlar']

## 3. Parameters are loop variables

Look at the URL of a filtered search:

```
https://www.emlakjet.com/satilik-konut/istanbul-fatih/?max_fiyat=3250000&min_fiyat=1500000
              path                    district         parameters
```

The district sits in the path, the price bounds in the query string. Both are just
strings you can build — which is what turns a single page into a dataset.

In [6]:
url = 'https://www.emlakjet.com/satilik-konut/istanbul-fatih/?max_fiyat=3250000&min_fiyat=1500000'

html = requests.get(url, headers=HEADERS, timeout=20).content
sel = Selector(text=html)

print("You have found:", len(sel.xpath('//*')), "elements")

You have found: 4281 elements


Two XPaths copied from the browser, for two different listing cards:

```
//*[@id="content-wrapper"]/div[1]/div[4]/div[2]/div[3]/div[2]/div/a/div[2]/div[1]/h3/text()
//*[@id="content-wrapper"]/div[1]/div[4]/div[2]/div[3]/div[3]/div/a/div[2]/div[1]/h3/text()
```

They differ in exactly one index. Drop that index and the path stops describing
*one card* and starts describing *every card* — this is the whole trick.

In [7]:
xpath_for_titles_of_adverts = ('//*[@id="content-wrapper"]/div[1]/div[4]/div[2]'
                               '/div[3]/div/div/a/div[2]/div[1]/h3/text()')

titles_of_adverts = sel.xpath(xpath_for_titles_of_adverts).extract()
print(len(titles_of_adverts), "titles")
titles_of_adverts[:5]

0 titles


[]

## 4. Clean what comes back

Raw extractions carry leftover markup, non-breaking spaces and the empty strings a
grid layout leaves behind. Clean once, in a function, so every collection step in this
notebook is cleaned the same way.

In [8]:
import re

def strip_tags(text):
    """Remove any residual HTML tags from an extracted string."""
    return re.sub(re.compile('<.*?>'), '', text)

def clean_data(data):
    """Strip tags and whitespace, drop empties. Returns a list of strings."""
    out = []
    for x in data:
        x = strip_tags(x)
        x = x.replace('\xa0', ' ')      # non-breaking space
        x = re.sub(r'\s+', ' ', x).strip()  # collapse whitespace
        if x:
            out.append(x)
    return out

cleaned_strings = clean_data(titles_of_adverts)
print(len(cleaned_strings))
cleaned_strings[:5]

0


[]

## 5. Build the URL from user input

In [ ]:
district = input("district (e.g. fatih): ")
maxim = input("max price: ")
minim = input("min price: ")

url = ('https://www.emlakjet.com/satilik-konut/istanbul-' + district +
       '/?max_fiyat=' + maxim + '&min_fiyat=' + minim)

html = requests.get(url, headers=HEADERS, timeout=20).content
sel = Selector(text=html)

titles_of_adverts = sel.xpath(xpath_for_titles_of_adverts).extract()
clean_data(titles_of_adverts)

## 6. Loop over districts and price bands

Why loop over price bands at all? Because the site paginates: one query returns only
the first page of results. Slicing the price range into intervals is a cheap way to
get more of the population without writing a pagination handler.

Note the `time.sleep()`. One request per second is the floor of polite scraping — and
it is also what keeps you from being blocked halfway through a collection.

In [ ]:
import time

def fetch_selector(url, pause=1.0):
    """Download a page and return a Selector, or None if the request failed."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print("failed:", url, "|", e)
        return None
    finally:
        time.sleep(pause)          # be polite, always
    return Selector(text=r.content)


def build_url(district, min_price, max_price):
    return ('https://www.emlakjet.com/satilik-konut/istanbul-' + district +
            '/?max_fiyat=' + str(max_price) + '&min_fiyat=' + str(min_price))

In [ ]:
district_list = ["uskudar", "fatih", "umraniye"]
intervals = [0, 5000000, 10000000, 20000000]

XPATHS = {
    'title': ('//*[@id="content-wrapper"]/div[1]/div[4]/div[2]'
              '/div[3]/div/div/a/div[2]/div[1]/h3/text()'),
    'link':  '//*[@id="content-wrapper"]/div[1]/div[4]/div[2]/div/div/div/a/@href',
    'price': ("//*[@id='content-wrapper']/div[1]/div[4]/div[2]"
              "/div[3]/div/div/a/div[2]/div[4]/div/div/span/text()[1]"),
}

output_titles, output_links, output_prices = [], [], []

for district in district_list:
    for i in range(len(intervals) - 1):
        url = build_url(district, intervals[i], intervals[i + 1])
        sel = fetch_selector(url)
        if sel is None:
            continue

        titles = sel.xpath(XPATHS['title']).extract()
        links  = sel.xpath(XPATHS['link']).extract()
        prices = sel.xpath(XPATHS['price']).extract()

        # Guard: if the three lists disagree, the page layout is not what we assumed.
        print(f"{district:10s} {intervals[i]:>9}-{intervals[i+1]:<9} "
              f"titles={len(titles)} links={len(links)} prices={len(prices)}")

        output_titles += titles
        output_links  += links
        output_prices += prices

In [ ]:
output_titles = clean_data(output_titles)
output_prices = clean_data(output_prices)

# Relative hrefs need the domain prepended before they are usable
BASE = "https://www.emlakjet.com"
adverts_link_merged = [BASE + i for i in output_links]

print(len(output_titles), len(adverts_link_merged), len(output_prices))

**Read that output before moving on.** If the three counts differ, you cannot put
them in the same DataFrame — the rows would not line up, and you would silently pair
a title with somebody else's price. When counts disagree, the fix is to extract per
*card* (loop over the listing containers, pull the three fields inside each one)
rather than field by field across the page.

In [ ]:
import pandas as pd

n = min(len(output_titles), len(adverts_link_merged), len(output_prices))
data_real_estate = pd.DataFrame({
    'title': output_titles[:n],
    'price': output_prices[:n],
    'link':  adverts_link_merged[:n],
})
data_real_estate.head()

## 7. Can we go further?

The results page does not carry the building age — the listing page does. So we follow
each link and scrape a second level. This is where a real crawl begins, and where a
Scrapy `Spider` (see the slides) earns its keep: it queues, retries and throttles the
requests for you.

Keep the loop small while you are testing. `range(5)`, not `range(len(df))`.

In [ ]:
xpath_for_age = '//*[@id="ilan-hakkinda"]/div/div[1]/ul'

ages = []
for i in range(5):                       # start small!
    print(f"progress: {100*i/5:.0f}%")
    sel = fetch_selector(data_real_estate['link'][i], pause=1.5)
    if sel is None:
        ages.append(None)
        continue
    block = sel.xpath(xpath_for_age).extract()
    ages.append(clean_data(block)[0] if block else None)

ages

In [ ]:
subset = data_real_estate.head(5).copy()
subset["details"] = ages
subset

In [ ]:
# Always write your collection to disk, dated. Re-running a scrape is expensive;
# re-reading a CSV is free.
from datetime import date
data_real_estate.to_csv(f"emlakjet_{date.today()}.csv", index=False)

---
# Part II — Application Programming Interfaces

A scraper reads a page written for humans. An API answers a question asked by a
program: you send parameters, you get JSON back. Same `requests` library, far less
guesswork — but the platform decides what exists.

Anatomy of a call:

| Piece | What it is |
|---|---|
| **endpoint** | the address of one resource, e.g. `.../youtube/v3/search` |
| **parameters** | your query as key–value pairs, passed via `params=` |
| **authentication** | an API key or OAuth token identifying you |
| **response** | JSON — nested dicts and lists — plus a status code |

## 8. A request without a key

Many scholarly and public APIs need no authentication at all. Crossref, below, returns
metadata for every DOI it knows about.

In [ ]:
import requests

r = requests.get(
    "https://api.crossref.org/works",
    params={"query.bibliographic": "computational social science",
            "rows": 5,
            "select": "title,author,issued,DOI"},
    headers={"User-Agent": "ADA-course/1.0 (mailto:mkina@ku.edu.tr)"},
    timeout=20,
)

print(r.status_code)      # 200 is the only one you want
data = r.json()           # JSON text -> Python dict
type(data)

In [ ]:
# JSON is nested. Walk it one level at a time until you find the list you want.
print(data.keys())
items = data["message"]["items"]
print(len(items), "items")
items[0]

In [ ]:
import pandas as pd

rows = []
for it in items:
    rows.append({
        "doi": it.get("DOI"),
        "title": (it.get("title") or [None])[0],
        "year": it.get("issued", {}).get("date-parts", [[None]])[0][0],
        "n_authors": len(it.get("author", [])),
    })

pd.DataFrame(rows)

Notice the `.get()` calls. Real API responses have missing fields; `data["author"]`
raises a `KeyError` on the first record without one and kills your loop after 400
successful requests. `.get()` with a default is not defensive style, it is the normal
way to read JSON.

## 9. Status codes, quotas, pagination

| Code | Meaning | What to do |
|---|---|---|
| 200 | fine | carry on |
| 401 / 403 | bad key, or quota exhausted | stop; check credentials and usage |
| 404 | wrong endpoint | re-read the docs |
| 429 | too many requests | back off and retry more slowly |
| 5xx | their problem | retry a few times, then give up |

Most APIs return results a **page** at a time and hand you a token or offset for the
next one. The pattern is always the same: loop while a token exists, and cap the loop
so a bug cannot burn your daily quota.

In [ ]:
import time

def paged_get(endpoint, params, page_key="pageToken", token_field="nextPageToken",
              max_pages=5, pause=0.5):
    """Generic pagination loop. Yields one parsed JSON page at a time."""
    params = dict(params)
    for page in range(max_pages):
        r = requests.get(endpoint, params=params, timeout=20)
        if r.status_code != 200:
            print("stopped at page", page, "| status", r.status_code, "|", r.text[:200])
            return
        payload = r.json()
        yield payload
        token = payload.get(token_field)
        if not token:
            return                      # no more pages
        params[page_key] = token
        time.sleep(pause)

---
# Part III — Use case: the YouTube Data API v3

### Getting a key (five minutes, free)

1. Go to `console.cloud.google.com` and create a project.
2. **APIs & Services → Library → YouTube Data API v3 → Enable**.
3. **Credentials → Create credentials → API key**. Copy it.
4. Store it *outside* the notebook — see the next cell.

### Quota arithmetic — read this before you loop

You get **10,000 units per day**, and calls are not priced equally:

| Call | Cost | Returns |
|---|---|---|
| `search.list` | **100 units** | up to 50 video ids per page |
| `videos.list` | 1 unit | metadata for up to 50 ids at once |
| `commentThreads.list` | 1 unit | up to 100 comments per page |

So 100 searches exhaust the day, while 100 metadata calls cost 1% of it. The design
follows directly: **search once, then work in batches of 50 ids.**

In [ ]:
import os

# Preferred: set the variable in your shell / .env, never in the notebook.
#   macOS/Linux:  export YT_API_KEY="AIza..."
#   Windows:      setx YT_API_KEY "AIza..."
API_KEY = os.environ.get("YT_API_KEY")

if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("YouTube API key: ")   # typed, not stored, not committed

BASE = "https://www.googleapis.com/youtube/v3" 

## 10. Step one — `search.list`: find the videos

In [ ]:
import requests, time, pandas as pd

def yt_search(query, max_pages=2, published_after=None, region="TR", lang="tr"):
    """Return a list of video ids for a query. Costs 100 units PER PAGE."""
    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": 50,            # the ceiling; asking for less wastes the 100 units
        "order": "relevance",        # or 'date', 'viewCount'
        "regionCode": region,
        "relevanceLanguage": lang,
        "key": API_KEY,
    }
    if published_after:
        params["publishedAfter"] = published_after   # e.g. "2024-01-01T00:00:00Z"

    ids = []
    for page in range(max_pages):
        r = requests.get(f"{BASE}/search", params=params, timeout=20)
        if r.status_code != 200:
            print("status", r.status_code, r.text[:300])
            break
        payload = r.json()
        ids += [it["id"]["videoId"] for it in payload.get("items", [])]
        token = payload.get("nextPageToken")
        if not token:
            break
        params["pageToken"] = token
        time.sleep(0.5)
    return ids


video_ids = yt_search("kentsel dönüşüm", max_pages=2)
print(len(video_ids), "video ids")
video_ids[:5]

## 11. Step two — `videos.list`: metadata, 50 at a time

`search.list` gives you ids and a thin snippet. The counts you actually want to model
— views, likes, comments, duration — live on the `videos` endpoint, which takes a
comma-separated list of up to 50 ids for a single unit.

In [ ]:
def yt_video_details(ids):
    """Metadata for a list of video ids. 1 unit per batch of 50."""
    rows = []
    for i in range(0, len(ids), 50):
        batch = ids[i:i + 50]
        r = requests.get(f"{BASE}/videos", timeout=20, params={
            "part": "snippet,statistics,contentDetails",
            "id": ",".join(batch),
            "key": API_KEY,
        })
        if r.status_code != 200:
            print("status", r.status_code, r.text[:300])
            break
        for it in r.json().get("items", []):
            sn, st = it.get("snippet", {}), it.get("statistics", {})
            rows.append({
                "video_id": it["id"],
                "title": sn.get("title"),
                "channel": sn.get("channelTitle"),
                "channel_id": sn.get("channelId"),
                "published_at": sn.get("publishedAt"),
                "duration": it.get("contentDetails", {}).get("duration"),
                # statistics arrive as STRINGS, and are absent when disabled
                "views": int(st["viewCount"]) if "viewCount" in st else None,
                "likes": int(st["likeCount"]) if "likeCount" in st else None,
                "comments": int(st["commentCount"]) if "commentCount" in st else None,
            })
        time.sleep(0.3)
    return pd.DataFrame(rows)


videos = yt_video_details(video_ids)
videos["published_at"] = pd.to_datetime(videos["published_at"])
print(videos.shape)
videos.head()

In [ ]:
# A first look — is this a corpus worth analysing?
print(videos[["views", "likes", "comments"]].describe())
videos.sort_values("views", ascending=False).head(10)[["title", "channel", "views"]]

## 12. Step three — `commentThreads.list`: the text

One row per comment, with the video it belongs to. This is the shape you want for
anything downstream: sentiment, topic models, or the LLM annotation you will do in
Module IV.

In [ ]:
def yt_comments(video_id, max_pages=2):
    """Top-level comments for one video. 1 unit per page of up to 100."""
    params = {
        "part": "snippet",
        "videoId": video_id,
        "maxResults": 100,
        "textFormat": "plainText",
        "order": "relevance",
        "key": API_KEY,
    }
    rows = []
    for page in range(max_pages):
        r = requests.get(f"{BASE}/commentThreads", params=params, timeout=20)
        if r.status_code == 403:
            # comments disabled for this video — common, and not an error in your code
            return rows
        if r.status_code != 200:
            print(video_id, "status", r.status_code)
            return rows
        payload = r.json()
        for it in payload.get("items", []):
            c = it["snippet"]["topLevelComment"]["snippet"]
            rows.append({
                "video_id": video_id,
                "comment_id": it["id"],
                "author": c.get("authorDisplayName"),
                "text": c.get("textDisplay"),
                "likes": c.get("likeCount"),
                "published_at": c.get("publishedAt"),
                "n_replies": it["snippet"].get("totalReplyCount"),
            })
        token = payload.get("nextPageToken")
        if not token:
            break
        params["pageToken"] = token
        time.sleep(0.3)
    return rows

In [ ]:
all_comments = []
for vid in videos["video_id"].head(10):        # keep it small on the first run
    all_comments += yt_comments(vid, max_pages=1)

comments = pd.DataFrame(all_comments)
print(comments.shape)
comments.head()

In [ ]:
# Join the comment text to its video and channel attributes — one analysable table
merged = comments.merge(
    videos[["video_id", "title", "channel", "views", "published_at"]],
    on="video_id", how="left", suffixes=("_comment", "_video"),
)

merged.to_csv(f"youtube_comments_{date.today()}.csv", index=False)
merged.head()